# Evaluate LAMPS Full Pipeline trên D2 (paper §5.2 / §5.3)

Yêu cầu trên Drive:
- `NT230/data/d1/saved_models/checkpoint-best-acc/model.bin`
- `NT230/data/d2/files.jsonl`
- `NT230/data/d2/packages.jsonl`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, sys, json
from pathlib import Path
from collections import defaultdict

DRIVE_D1 = '/content/drive/My Drive/NT230/data/d1'
DRIVE_D2 = '/content/drive/My Drive/NT230/data/d2'

!git clone --depth=1 https://github.com/khoilv2005/NT230.git /content/NT230
sys.path.insert(0, '/content/NT230/src')

# model.bin từ d1
os.makedirs('/content/saved_models/checkpoint-best-acc', exist_ok=True)
shutil.copy(f'{DRIVE_D1}/saved_models/checkpoint-best-acc/model.bin',
            '/content/saved_models/checkpoint-best-acc/model.bin')
print('✅ model.bin:', round(os.path.getsize('/content/saved_models/checkpoint-best-acc/model.bin')/1e6), 'MB')

# D2 data
shutil.copy(f'{DRIVE_D2}/files.jsonl',    '/content/d2_files.jsonl')
shutil.copy(f'{DRIVE_D2}/packages.jsonl', '/content/d2_packages.jsonl')
print('✅ d2_files.jsonl:   ', sum(1 for _ in open('/content/d2_files.jsonl')), 'records')
print('✅ d2_packages.jsonl:', sum(1 for _ in open('/content/d2_packages.jsonl')), 'records')

In [ ]:
!pip install -q transformers==4.40.0 torch scikit-learn scipy pandas tqdm

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ No GPU')

In [ ]:
from lamps.agents.classifier import ClassifierAgent, FileClassification
from lamps.agents.extractor import ExtractedFile
from lamps.agents.verdict import VerdictAgent
from lamps.evaluation.metrics import classification_report, format_report
from lamps.utils import read_jsonl

classifier    = ClassifierAgent(checkpoint='/content/saved_models/checkpoint-best-acc/model.bin', batch_size=64)
verdict_agent = VerdictAgent(llm=None)
print('✅ Agents loaded')

In [ ]:
file_records    = list(read_jsonl(Path('/content/d2_files.jsonl')))
package_records = list(read_jsonl(Path('/content/d2_packages.jsonl')))
print(f'Files: {len(file_records)} | Packages: {len(package_records)}')

In [ ]:
# Extractor Agent — rule-based filter
NOISY = {'tests','test','testing','docs','doc','examples','_vendor','vendor'}
def is_relevant(path):
    parts = str(path).lower().replace('\\','/').split('/')
    return not any(p in NOISY for p in parts) and not parts[-1].startswith('test_')

filtered = [r for r in file_records if is_relevant(r.get('path', ''))]
print(f'After filter: {len(filtered)} / {len(file_records)} files')

In [ ]:
# Classifier Agent
files = [ExtractedFile(package=str(r['package']), path=Path('<memory>'),
                       rel_path=str(r.get('path','')), source=str(r['func']))
         for r in filtered]

print(f'Classifying {len(files)} files...')
classifications = classifier.classify_files(files)

y_file_true = [int(r['target']) for r in filtered]
y_file_pred = [c.target for c in classifications]
print('\n=== File-level ===')
print(format_report(classification_report(y_file_true, y_file_pred)))

In [ ]:
# Verdict Agent — conservative aggregation
cls_by_pkg = defaultdict(list)
for cls in classifications:
    cls_by_pkg[cls.package].append(cls)

y_pkg_true, y_pkg_pred, pkg_preds = [], [], []
for pkg in package_records:
    verdict = verdict_agent.aggregate(pkg['package'], cls_by_pkg.get(pkg['package'], []))
    y_pkg_true.append(int(pkg['label']))
    y_pkg_pred.append(verdict.target)
    pkg_preds.append({'package': pkg['package'], 'target': int(pkg['label']),
                      'predicted': verdict.target, 'n_malicious_files': len(verdict.malicious_files)})

pkg_report = classification_report(y_pkg_true, y_pkg_pred)
print('\n=== Package-level (paper Table 3) ===')
print(format_report(pkg_report))

In [ ]:
# Save về Drive
os.makedirs('/content/results_d2', exist_ok=True)
with open('/content/results_d2/package_report.json','w') as f:
    json.dump(pkg_report.to_dict(), f, indent=2)
with open('/content/results_d2/package_predictions.jsonl','w') as f:
    f.write('\n'.join(json.dumps(p) for p in pkg_preds))

shutil.copytree('/content/results_d2', f'{DRIVE_D2}/results', dirs_exist_ok=True)
print(f'✅ Saved to Drive: NT230/data/d2/results/')

In [ ]:
# Export wrong predictions: file-level and package-level FP/FN
from pathlib import Path
import csv

OUTPUT_DIR = Path('/content/results_d2')
DRIVE_OUTPUT_DIR = Path(DRIVE_D2) / 'results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def label_name(x):
    return 'malicious' if int(x) == 1 else 'benign'

def write_jsonl(path, rows):
    path.write_text(
        '\n'.join(json.dumps(r, ensure_ascii=False) for r in rows) + ('\n' if rows else ''),
        encoding='utf-8',
    )

def write_csv(path, rows, fieldnames):
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

# File-level errors: each row is one filtered source file that CodeBERT classified.
file_predictions = []
for r, c in zip(filtered, classifications):
    file_predictions.append({
        'idx': str(r.get('idx', '')),
        'package': str(r['package']),
        'path': str(r.get('path', '')),
        'target': int(r['target']),
        'target_label': label_name(r['target']),
        'predicted': int(c.target),
        'predicted_label': label_name(c.target),
        'score': float(c.score),
        'source_chars': len(str(r.get('func', ''))),
    })

file_false_positives = []  # benign -> malicious
file_false_negatives = []  # malicious -> benign
for row in file_predictions:
    if int(row['target']) == int(row['predicted']):
        continue
    item = {**row, 'error_type': 'FP' if int(row['target']) == 0 else 'FN'}
    if item['error_type'] == 'FP':
        file_false_positives.append(item)
    else:
        file_false_negatives.append(item)

file_false_negatives_sorted = sorted(file_false_negatives, key=lambda r: float(r['score']))
file_false_positives_sorted = sorted(file_false_positives, key=lambda r: -float(r['score']))
file_wrong_sorted = file_false_negatives_sorted + file_false_positives_sorted

# Package-level errors: each row is one package after Verdict aggregation.
package_meta = {r['package']: r for r in package_records}
package_predictions = []
for row in pkg_preds:
    meta = package_meta.get(row['package'], {})
    package_predictions.append({
        **row,
        'target_label': label_name(row['target']),
        'predicted_label': label_name(row['predicted']),
        'dataset_n_files': int(meta.get('n_files', 0)),
        'dataset_files': '|'.join(meta.get('files', [])[:20]) if isinstance(meta.get('files', []), list) else '',
    })

package_false_positives = []  # benign -> malicious
package_false_negatives = []  # malicious -> benign
for row in package_predictions:
    if int(row['target']) == int(row['predicted']):
        continue
    item = {**row, 'error_type': 'FP' if int(row['target']) == 0 else 'FN'}
    if item['error_type'] == 'FP':
        package_false_positives.append(item)
    else:
        package_false_negatives.append(item)

package_false_negatives_sorted = sorted(package_false_negatives, key=lambda r: (-int(r.get('dataset_n_files', 0)), r['package']))
package_false_positives_sorted = sorted(package_false_positives, key=lambda r: (-int(r.get('n_malicious_files', 0)), -int(r.get('dataset_n_files', 0)), r['package']))
package_wrong_sorted = package_false_negatives_sorted + package_false_positives_sorted

file_fields = [
    'idx', 'package', 'path', 'error_type', 'target', 'target_label', 'predicted', 'predicted_label',
    'score', 'source_chars',
]
package_fields = [
    'package', 'error_type', 'target', 'target_label', 'predicted', 'predicted_label',
    'dataset_n_files', 'n_malicious_files', 'dataset_files',
]

for out_dir in (OUTPUT_DIR, DRIVE_OUTPUT_DIR):
    write_jsonl(out_dir / 'file_wrong_predictions.jsonl', file_wrong_sorted)
    write_jsonl(out_dir / 'file_false_negatives.jsonl', file_false_negatives_sorted)
    write_jsonl(out_dir / 'file_false_positives.jsonl', file_false_positives_sorted)
    write_csv(out_dir / 'file_wrong_predictions.csv', file_wrong_sorted, file_fields)
    write_csv(out_dir / 'file_false_negatives.csv', file_false_negatives_sorted, file_fields)
    write_csv(out_dir / 'file_false_positives.csv', file_false_positives_sorted, file_fields)

    write_jsonl(out_dir / 'package_wrong_predictions.jsonl', package_wrong_sorted)
    write_jsonl(out_dir / 'package_false_negatives.jsonl', package_false_negatives_sorted)
    write_jsonl(out_dir / 'package_false_positives.jsonl', package_false_positives_sorted)
    write_csv(out_dir / 'package_wrong_predictions.csv', package_wrong_sorted, package_fields)
    write_csv(out_dir / 'package_false_negatives.csv', package_false_negatives_sorted, package_fields)
    write_csv(out_dir / 'package_false_positives.csv', package_false_positives_sorted, package_fields)

    (out_dir / 'error_summary.json').write_text(json.dumps({
        'file_wrong': len(file_wrong_sorted),
        'file_false_positives': len(file_false_positives_sorted),
        'file_false_negatives': len(file_false_negatives_sorted),
        'package_wrong': len(package_wrong_sorted),
        'package_false_positives': len(package_false_positives_sorted),
        'package_false_negatives': len(package_false_negatives_sorted),
        'file_wrong_predictions_csv': str(out_dir / 'file_wrong_predictions.csv'),
        'package_wrong_predictions_csv': str(out_dir / 'package_wrong_predictions.csv'),
    }, indent=2), encoding='utf-8')

summary = {
    'file_wrong': len(file_wrong_sorted),
    'file_false_positives': len(file_false_positives_sorted),
    'file_false_negatives': len(file_false_negatives_sorted),
    'package_wrong': len(package_wrong_sorted),
    'package_false_positives': len(package_false_positives_sorted),
    'package_false_negatives': len(package_false_negatives_sorted),
    'drive_output': str(DRIVE_OUTPUT_DIR),
}
print(json.dumps(summary, indent=2))

print('\nTop 20 file false negatives (malware files missed):')
for r in file_false_negatives_sorted[:20]:
    print(f"FN score={float(r['score']):.4f} package={r['package']} path={r['path']}")

print('\nTop 20 file false positives (benign files false alarm):')
for r in file_false_positives_sorted[:20]:
    print(f"FP score={float(r['score']):.4f} package={r['package']} path={r['path']}")

print('\nTop 20 package false negatives (malware packages missed):')
for r in package_false_negatives_sorted[:20]:
    print(f"FN package={r['package']} files={r['dataset_n_files']} malicious_files={r['n_malicious_files']}")

print('\nTop 20 package false positives (benign packages false alarm):')
for r in package_false_positives_sorted[:20]:
    print(f"FP package={r['package']} files={r['dataset_n_files']} malicious_files={r['n_malicious_files']}")